In [13]:
import sqlite3
import os
import pandas as pd


In [14]:
# get the data

base_path = "/Users/jedmeier/Projects/fast-trade/archive"

files = [
    f"{base_path}/bnus_BTCUSD_2019.csv",
    f"{base_path}/bnus_BTCUSD_2020.csv",
    f"{base_path}/bnus_BTCUSD_2021.csv",
    f"{base_path}/bnus_BTCUSD_2022.csv",
    f"{base_path}/bnus_BTCUSD_2023.csv"
]

dfs = []

for filename in files:
    df = pd.read_csv(filename)
    df.date = pd.to_datetime(df.date, unit="s")
    dfs.append(df)


master_df = pd.concat(dfs)
print(master_df)

import datetime

def str_to_datetime(s):
  split = s.split('-')
  year, month, day = int(split[0]), int(split[1]), int(split[2])
  return datetime.datetime(year=year, month=month, day=day)


                      date      low      high      open     close   volume  \
0      2019-09-17 10:17:00  10170.0  10170.00  10170.00  10170.00  0.00200   
1      2019-09-17 10:18:00  10170.0  10170.00  10170.00  10170.00  0.00000   
2      2019-09-17 10:19:00  10170.0  10170.00  10170.00  10170.00  0.00000   
3      2019-09-17 10:20:00  10170.0  10170.00  10170.00  10170.00  0.00000   
4      2019-09-17 10:21:00  10170.0  10170.00  10170.00  10170.00  0.00000   
...                    ...      ...       ...       ...       ...      ...   
279126 2023-07-14 02:57:00  24920.0  25055.72  24979.35  24920.00  0.84385   
279127 2023-07-14 02:58:00  24900.0  25055.72  24920.00  25054.98  0.18515   
279128 2023-07-14 02:59:00  24990.0  25100.67  24990.00  25073.21  0.70879   
279129 2023-07-14 02:59:00  24990.0  25100.67  24990.00  25073.21  0.70879   
279130 2023-07-14 02:59:00  24990.0  25100.67  24990.00  25073.21  0.70879   

        quote_asset_volume  number_of_trades  taker_buy_base_as

In [16]:
def df_to_windowed_df(dataframe, first_date_str, last_date_str, n=3):
  first_date = str_to_datetime(first_date_str)
  last_date  = str_to_datetime(last_date_str)

  target_date = first_date
  
  dates = []
  X, Y = [], []

  last_time = False
  while True:
    df_subset = dataframe.loc[:target_date].tail(n+1)
    
    if len(df_subset) != n+1:
      print(f'Error: Window of size {n} is too large for date {target_date}')
      return

    values = df_subset['Close'].to_numpy()
    x, y = values[:-1], values[-1]

    dates.append(target_date)
    X.append(x)
    Y.append(y)

    next_week = dataframe.loc[target_date:target_date+datetime.timedelta(days=7)]
    next_datetime_str = str(next_week.head(2).tail(1).index.values[0])
    next_date_str = next_datetime_str.split('T')[0]
    year_month_day = next_date_str.split('-')
    year, month, day = year_month_day
    next_date = datetime.datetime(day=int(day), month=int(month), year=int(year))
    
    if last_time:
      break
    
    target_date = next_date

    if target_date == last_date:
      last_time = True
    
  ret_df = pd.DataFrame({})
  ret_df['Target Date'] = dates
  
  X = np.array(X)
  for i in range(0, n):
    X[:, i]
    ret_df[f'Target-{n-i}'] = X[:, i]
  
  ret_df['Target'] = Y

  return ret_df

windowed_df = df_to_windowed_df(df, 
                                '2029-03-25', 
                                '2023-05-23', 
                                n=3)
windowed_df

TypeError: '<' not supported between instances of 'int' and 'datetime.datetime'

In [7]:
def windowed_df_to_date_X_y(windowed_dataframe):
  df_as_np = windowed_dataframe.to_numpy()

  dates = df_as_np[:, 0]

  middle_matrix = df_as_np[:, 1:-1]
  X = middle_matrix.reshape((len(dates), middle_matrix.shape[1], 1))

  Y = df_as_np[:, -1]

  return dates, X.astype(np.float32), Y.astype(np.float32)

dates, X, y = windowed_df_to_date_X_y(windowed_df)

dates.shape, X.shape, y.shape

NameError: name 'windowed_df' is not defined

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers

model = Sequential([layers.Input((3, 1)),
                    layers.LSTM(64),
                    layers.Dense(32, activation='relu'),
                    layers.Dense(32, activation='relu'),
                    layers.Dense(1)])

model.compile(loss='mse', 
              optimizer=Adam(learning_rate=0.001),
              metrics=['mean_absolute_error'])

model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100)
# just sta

NameError: name 'X_train' is not defined